# Негативные рёбра: «поглощающая стена» вместо «штрафа вперёд»

> **Запуск** — `scripts/run_demo.sh`, либо вручную, без установки чего-либо в систему:
> ```
> uv run --no-project --with matplotlib --with ipywidgets --with jupyterlab \
>     jupyter lab scripts/negative_edges_demo.ipynb
> ```
> uv собирает эфемерное окружение и выбрасывает его после выхода.

> **Объём фичи.** Фаза 1 — только собственные стены эго. Переключатель «наследованное недоверие»
> показывает отложенную фазу 2 (`Intersubjective/meritrank-rust#86`). Минус кодируется отрицательным
> весом ребра, сила стены $d = \min(|w|, 1)$. Требования и глоссарий (русские термины ↔ английские) —
> `NEGATIVE_EDGES_FEATURE.md`, обоснование решений — `NEGATIVE_EDGES_JOURNAL.md`.

Демонстрация на линейной цепочке доверия. По оси **x** — номер персонажа в цепочке, по оси **y** —
уровень доверия эго (Алисы) к нему, в долях от базового доверия к первому другу.

**Топология** — две цепочки, обе висят на эго:

```
Алиса ─→ a1 ─→ a2 ─→ … ─→ aK ─→ [Боб ✗] ─→ t1 ─→ … ─→ tm     цепочка, ведущая к Бобу
Алиса ─→ c1 ─→ c2 ─→ … ─→ cj ─→ …                             контрольная цепочка
                             ╰────⊣ Боб                       минус, если обвинитель — не Алиса
```

Все рёбра положительные и единичного веса. Кто-то ставит **минус Бобу** (позиция `K+1`) —
либо сама Алиса, либо доверенный ей узел `cj`, см. «источник недоверия» ниже.
Узлы `t_j` — это **исходящие рёбра самого Боба**: те, кого он целенаправленно хвалит.
Контрольная цепочка ни с Бобом, ни с его подопечными не связана — она нужна, чтобы видеть
побочный ущерб для непричастных.

## Что сравнивается

**Текущая модель** (`graph.rs::generate_walk_segment`): минус — это *проходимое ребро*.
Walk сворачивает в Боба и дальше идёт по **его** плюсам, штрафуя всё, что встретит.
Множество наказанных выбирает Боб → это и есть curse-attack.

**Новая модель**: минус — это *свойство узла в рамке эго*. Негативные рёбра непроходимы,
Боб — поглощающая стена. Замкнутая форма для узла C:

$$\text{score}(C) \;=\; n_C \cdot \bigl(1 - (1+\lambda)\,q_C\bigr)$$

где $n_C$ — ожидаемое число визитов в C за одно блуждание, $q_C$ — вероятность, что блуждание,
дойдя до C, дальше будет **поглощено** стеной. Не просто дойдёт до недоверенного узла: у мягкой
стены ($d<1$) это разные вещи. Ключевое: $q_C$ — **собственный выбор C**,
Боб на него влиять не может.

## Два разных «один шаг» — не путать

Термин «1-step distrust» перегружен: у Guha et al. 2004 это про **глубину наследования**
недоверия, а «наказать только прямого поручителя» — это про **радиус вины**. Оси ортогональны
и включаются независимо.

| | **Наследование недоверия** (Guha 2004) | **Радиус вины** |
|---|---|---|
| Вопрос | *чьи* минусы попадают в рамку Алисы | *кого* наказать за то, что привёл к Бобу |
| Направление | вперёд по доверию: Алиса → `cj`, `cj` ⊣ Боб | назад по блужданию: … → `aK` → Боб |
| «один шаг» значит | наследуем минусы тех, кому доверяем, и **не идём дальше**: минусы самого Боба в рамку Алисы уже не попадают | штраф только прямому поручителю `aK`, а не всему префиксу |
| Переключатель | «источник недоверия» | «вина только прямому поручителю» |

Наследование здесь именно одношаговое по Guha: Алиса подхватывает минус узла `cj`, которому
доверяет, но цепочка на этом обрывается. Сила наследованного недоверия пропорциональна доверию
Алисы к обвинителю: $d = s \cdot \dfrac{\text{доверие к } c_j}{\text{доверие к } c_1}$ — чем
дальше обвинитель, тем мягче стена. Обвинитель при этом **не наказывается**: он на контрольной
цепочке и ни в один отравленный префикс не попадает.

Наследование считается **отдельным проходом** (журнал, D6): сначала строится множество стен $D_A$
рамки Алисы, потом идут блуждания. Поэтому стена на Бобе действует на все блуждания Алисы, а не
только на прошедшие через `cj`, а сам `cj` платит лишь за собственную маршрутизацию к Бобу — здесь
она нулевая. Формула для $d$ — одна из возможных функций веса по входящему доверию; требования к
наследованию — в `Intersubjective/meritrank-rust#86`.

## Переключатели

| Режим | Что делает |
|---|---|
| **источник недоверия** | минус ставит сама Алиса ($d=s$) или доверенный узел `cj` — наследование по Guha ($d = s\,\alpha^{\,j-1}$). |
| **обратная дискредитация** | $\lambda > 0$: узлы отравленного блуждания получают $-\lambda$ вместо $+1$. При $\lambda=0$ работает только стена. |
| **вина только прямому поручителю** | радиус вины = 1: штраф получают лишь $a_K$ и Боб. Иначе — весь префикс с затуханием $\gamma$ по расстоянию до Боба. |
| **ловушка нормировки** | делить на реально набранные хиты, как сейчас в `rank.rs::get_node_score`, строки 108–112, вместо фиксированного знаменателя. |

In [1]:
# Окружение поднимается через uv, см. заголовок выше или scripts/run_demo.sh
%matplotlib inline
import matplotlib.pyplot as plt

ALPHA, K0, M0 = 0.85, 6, 3      # значения по умолчанию
CTL_TAIL = 3                    # насколько контрольная цепочка длиннее позиции Боба

In [2]:
def dims(K, m):
    """B — позиция Боба; La, Lc — длины цепочки к Бобу и контрольной.

    Длина контрольной цепочки намеренно не зависит от m: иначе проверка аксиомы A1
    («меняет ли что-то то, кого Боб хвалит») ловила бы собственный артефакт топологии.
    """
    B = K + 1
    return B, B + m, B + CTL_TAIL


def baseline(alpha, K, m, trap):
    """Как выглядел бы граф, если бы Алиса не ставила минус вовсе."""
    B, La, Lc = dims(K, m)
    N = max(La, Lc)
    a = [0.0] * (N + 1)
    c = [0.0] * (N + 1)
    for i in range(1, La + 1):
        a[i] = 0.5 * alpha ** i          # эго делит поток поровну между двумя цепочками
    for i in range(1, Lc + 1):
        c[i] = 0.5 * alpha ** i
    Z = (1.0 + sum(a) + sum(c)) if trap else 1.0
    return [x / Z for x in a], [x / Z for x in c]


def accuser_pos(K, m, acc):
    """Позиция обвинителя на контрольной цепочке; у c_j должен остаться и свой преемник."""
    B, La, Lc = dims(K, m)
    return max(1, min(acc, Lc - 1))


def effective_d(alpha, K, m, s, inherit, acc):
    """Сила стены. Наследованное недоверие слабее ровно во столько раз,
    во сколько Алиса доверяет обвинителю меньше, чем ближайшему другу."""
    if not inherit:
        return s
    return s * alpha ** (accuser_pos(K, m, acc) - 1)


def current_model(alpha, K, m, wneg, inherit=False, acc=2):
    """Текущая семантика MeritRank: минус проходим, суффикс после него штрафуется.

    Нормировка здесь всегда «по реальным хитам» — так и написано в rank.rs::get_node_score.
    Наследование недоверия в этой модели не одношаговое: минус узла c_j — обычное ребро,
    блуждание сворачивает в него из любой точки, куда сумело дойти.
    """
    B, La, Lc = dims(K, m)
    N = max(La, Lc)
    ap = [0.0] * (N + 1); an = [0.0] * (N + 1); cp = [0.0] * (N + 1)

    if not inherit:                              # минус у эго: выбор среди a1, c1 и Боба
        pa = pc = 1.0 / (2.0 + wneg)
        pn = wneg / (2.0 + wneg)
        for i in range(1, La + 1):
            ap[i] = alpha ** i * pa
        for i in range(1, Lc + 1):
            cp[i] = alpha ** i * pc
        for k in range(0, m + 1):                # k = 0 — сам Боб
            an[B + k] = alpha ** (1 + k) * pn    # проклятие идёт по исходящим рёбрам Боба
    else:                                        # минус у c_j: эго делит поток поровну
        j = accuser_pos(K, m, acc)
        p1 = 1.0 / (1.0 + wneg)                  # у c_j теперь два ребра: c_{j+1} и минус Бобу
        pn = wneg / (1.0 + wneg)
        for i in range(1, La + 1):
            ap[i] = 0.5 * alpha ** i
        for i in range(1, Lc + 1):
            cp[i] = 0.5 * alpha ** i * (p1 if i > j else 1.0)   # хвост за обвинителем разбавлен
        for k in range(0, m + 1):
            an[B + k] = 0.5 * alpha ** (j + 1 + k) * pn

    Z = 1.0 + sum(ap) + sum(an) + sum(cp)
    return [(ap[i] - an[i]) / Z for i in range(N + 1)], [x / Z for x in cp]


def blame_weights(K, m, lam, blame_one_hop, gamma):
    """Доля вины узла за то, что он привёл блуждание к недоверенному узлу."""
    B, La, Lc = dims(K, m)
    b = [0.0] * (max(La, Lc) + 1)
    if lam <= 0.0:
        return b
    if blame_one_hop:
        b[B] = 1.0
        if K >= 1:
            b[K] = 1.0                           # только Боб и его прямой поручитель
    else:
        for i in range(1, B + 1):
            b[i] = gamma ** (B - i)              # весь префикс, с затуханием по расстоянию
    return b


def new_model(alpha, K, m, d, lam, blame_one_hop, gamma, trap):
    """Поглощающая стена + опциональная обратная дискредитация."""
    B, La, Lc = dims(K, m)
    N = max(La, Lc)
    va = [0.0] * (N + 1); q = [0.0] * (N + 1); vc = [0.0] * (N + 1)
    for i in range(1, B + 1):
        va[i] = 0.5 * alpha ** i
        q[i] = d * alpha ** (B - i)              # P(упереться в стену | дошли до i)
    for j in range(1, m + 1):
        va[B + j] = 0.5 * alpha ** (B + j) * (1.0 - d)    # при d=1 подопечные недостижимы
    for i in range(1, Lc + 1):
        vc[i] = 0.5 * alpha ** i
    pos = [va[i] * (1 - q[i]) for i in range(N + 1)]
    neg = [va[i] * q[i] for i in range(N + 1)]
    b = blame_weights(K, m, lam, blame_one_hop, gamma)
    raw = [pos[i] - lam * b[i] * neg[i] for i in range(N + 1)]
    Z = (1.0 + sum(pos) + sum(neg) + sum(vc)) if trap else 1.0
    return [x / Z for x in raw], [x / Z for x in vc]

In [3]:
def plot(alpha=ALPHA, K=K0, m=M0, s=1.0, lam=0.6, blame_one_hop=False, gamma=0.8,
         trap=False, backward=True, show_current=True, wneg=1.0,
         inherit=False, acc=2):
    lam = lam if backward else 0.0
    B, La, Lc = dims(K, m)
    xa, xc = list(range(1, La + 1)), list(range(1, Lc + 1))
    d = effective_d(alpha, K, m, s, inherit, acc)
    j = accuser_pos(K, m, acc)

    bA, bC = baseline(alpha, K, m, trap)
    nA, nC = new_model(alpha, K, m, d, lam, blame_one_hop, gamma, trap)
    cA, cC = current_model(alpha, K, m, wneg, inherit, acc)
    ref = bA[1]                                   # доверие к первому другу без минуса
    refc = baseline(alpha, K, m, True)[0][1]      # у текущей модели своя нормировка

    def sc(v, xs, r):
        return [v[i] / r for i in xs]

    fig, ax = plt.subplots(figsize=(11.5, 6.4))
    if m > 0:
        ax.axvspan(B + 0.5, La + 0.45, color='#d9534f', alpha=0.07)
        ax.text((B + 0.5 + La) / 2, 1.13, 'подопечные Боба\n(его исходящие рёбра)',
                ha='center', va='top', fontsize=8.5, color='#a33')
    ax.axhline(0, color='#333', lw=0.9)
    ax.axvline(B, color='#d9534f', lw=1.0, ls=':', alpha=0.8)
    ax.annotate('Боб ✗', xy=(B, 1.16), ha='center', fontsize=10.5,
                color='#d9534f', fontweight='bold')
    if inherit:
        ax.annotate(f'обвинитель c{j}\n(минус ставит он, а не Алиса)',
                    xy=(j, alpha ** (j - 1)), xytext=(j, 1.10), ha='center',
                    fontsize=8.5, color='#2ca02c',
                    arrowprops=dict(arrowstyle='->', color='#2ca02c', lw=1.1))

    # bA[i]/ref == alpha**(i-1) при любой нормировке; рисуем широкой лентой, чтобы
    # совпадающая с ней контрольная цепочка не прятала базовую линию
    xb = list(range(1, max(La, Lc) + 1))
    ax.plot(xb, [alpha ** (i - 1) for i in xb], lw=5, color='#bbb', alpha=0.55,
            solid_capstyle='round', label='без минуса (базовая линия)')
    ax.plot(xa, sc(nA, xa, ref), '-o', lw=2.4, ms=5, color='#1f77b4',
            label='новая модель: цепочка → Боб')
    ax.plot(xc, sc(nC, xc, ref), '-s', lw=1.8, ms=4, color='#2ca02c',
            label='новая модель: контрольная цепочка')
    if show_current:
        ax.plot(xa, sc(cA, xa, refc), ':^', lw=1.8, ms=5, color='#d62728',
                label='текущая модель: цепочка → Боб')
        ax.plot(xc, sc(cC, xc, refc), ':v', lw=1.4, ms=4, color='#ff7f0e',
                label='текущая модель: контрольная цепочка')

    ax.set_xticks(list(range(1, max(La, Lc) + 1)))
    ax.set_xlabel('позиция в цепочке   (эго — слева, за пределами оси)')
    ax.set_ylabel('доверие эго\n(1.0 = доверие к первому другу без минуса)')
    ax.set_title('Влияние негативного ребра на цепочку доверия', fontsize=12.5)
    ax.grid(alpha=0.25)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=3, fontsize=9,
              frameon=False)
    lo = min([0.0] + sc(nA, xa, ref) + (sc(cA, xa, refc) if show_current else []))
    ax.set_ylim(min(lo * 1.2, -0.06), 1.28)
    plt.tight_layout()
    plt.show()

    print(audit(alpha, K, m, d, lam, blame_one_hop, gamma, trap, wneg, inherit, acc))


def audit(alpha, K, m, d, lam, blame_one_hop, gamma, trap, wneg, inherit=False, acc=2):
    """Проверка аксиом при текущих настройках."""
    B, La, Lc = dims(K, m)
    bA, bC = baseline(alpha, K, m, trap)
    nA, nC = new_model(alpha, K, m, d, lam, blame_one_hop, gamma, trap)
    ref = bA[1]

    def sensitivity(f, r):
        """A1: сдвиг чужих оценок от того, что Боб добавил себе двух подопечных."""
        a1, c1 = f(m)
        a2, c2 = f(m + 2)
        rng = range(1, B + 1)
        return max(max(abs(a2[i] - a1[i]) for i in rng),
                   max(abs(c2[i] - c1[i]) for i in rng)) / r

    a1_new = sensitivity(lambda mm: new_model(alpha, K, mm, d, lam, blame_one_hop, gamma, trap), ref)
    a1_cur = sensitivity(lambda mm: current_model(alpha, K, mm, wneg, inherit, acc),
                         baseline(alpha, K, m, True)[0][1])

    # A6: минус не должен никому поднимать оценку
    gain = max(nC[i] / bC[i] - 1 for i in range(1, Lc + 1))

    ok = lambda c: '✓' if c else '✗'
    q1 = d * alpha ** K
    voucher = (nA[K] / bA[K]) if K >= 1 else 1.0
    soft = ('\n\nстена мягкая (d < 1): Боб сохраняет ' f'{(1 - d) * 100:.0f}% прохода и через них '
            'подкармливает своих подопечных.\nНа третьих лиц это не влияет — пока знаменатель '
            'фиксирован; вместе с ловушкой нормировки уже влияет.' if d < 1 - 1e-9 else '')
    free = '   ← ловушка: непричастные получили бонус' if gain > 1e-9 else ''
    src = (f'источник недоверия: обвинитель c{accuser_pos(K, m, acc)} '
           f'(наследование, Guha 1-step)  →  d = {d:.3f}'
           if inherit else f'источник недоверия: сама Алиса  →  d = {d:.3f}')
    return (
        f'\n{src}\n'
        '\nA1  иммунитет к curse-attack — влияют ли исходящие рёбра Боба на чужие оценки\n'
        f'      новая модель:  {ok(a1_new < 1e-9)}   сдвиг {a1_new * 100:6.2f}%\n'
        f'      текущая:       {ok(a1_cur < 1e-9)}   сдвиг {a1_cur * 100:6.2f}%\n'
        'A6  минус никому не поднимает оценку — контрольная цепочка\n'
        f'      новая модель:  {ok(gain <= 1e-9)}   макс. прирост {gain * 100:+6.2f}%{free}\n'
        f'\nq(a1) = {q1:.3f}  →  множитель влияния 1−(1+λ)q = {1 - (1 + lam) * q1:+.3f}'
        f'   |  прямой поручитель a{K}: ×{voucher:+.2f} от базового уровня'
        f'{soft}'
    )

In [4]:
import ipywidgets as w
from IPython.display import display

S = dict(style={'description_width': '225px'}, layout=w.Layout(width='440px'),
         continuous_update=True)
C = dict(indent=False, layout=w.Layout(width='440px'))

# ── ось 1: чьё недоверие попадает в рамку Алисы (наследование, Guha) ─────────────
inherit  = w.Checkbox(value=False, **C,
                      description='наследованное недоверие, Guha 1-step, фаза 2 (минус ставит cj, не Алиса)')
acc      = w.IntSlider(value=2, min=1, max=10, step=1,
                       description='позиция обвинителя cj', **S)
# ── ось 2: кого наказывать за то, что привёл к Бобу (радиус вины) ────────────────
backward = w.Checkbox(value=True,  description='обратная дискредитация (λ > 0)', **C)
blame_one_hop = w.Checkbox(value=False, **C,
                           description='вина только прямому поручителю (радиус = 1)')
# ── прочее ──────────────────────────────────────────────────────────────────────
trap         = w.Checkbox(value=False, description='ловушка нормировки (делить на реальные хиты)', **C)
show_current = w.Checkbox(value=True,  description='показать текущую модель (штраф вперёд)', **C)

alpha = w.FloatSlider(value=ALPHA, min=0.50, max=0.95, step=0.01,
                      description='α — вероятность продолжения блуждания', **S)
K     = w.IntSlider(value=K0, min=2, max=12, step=1,
                    description='длина цепочки до Боба', **S)
m     = w.IntSlider(value=M0, min=0, max=6, step=1,
                    description='подопечных у Боба', **S)
s     = w.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                      description='s — сила минуса (поглощение)', **S)
lam   = w.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.05,
                      description='λ — вес обратной дискредитации', **S)
gamma = w.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05,
                      description='γ — затухание вины (если радиус > 1)', **S)
wneg  = w.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1,
                      description='|w⁻| — вес минуса (текущая модель)', **S)

BIND = dict(alpha=alpha, K=K, m=m, s=s, lam=lam, blame_one_hop=blame_one_hop,
            gamma=gamma, backward=backward, inherit=inherit, acc=acc)
out = w.interactive_output(plot, dict(
    BIND, trap=trap, show_current=show_current, wneg=wneg))

display(w.VBox([
    w.HBox([w.VBox([inherit, blame_one_hop, backward, trap, show_current]),
            w.VBox([alpha, K, m, acc])]),
    w.HBox([w.VBox([s, lam]), w.VBox([gamma, wneg])]),
    out,
]))

## Что посмотреть

1. **Curse-attack.** Поставьте `подопечных у Боба` = 4. Красный пунктир (текущая модель) уходит
   в глубокий минус справа от Боба: он *сам выбрал*, кого утопить, просто расставив плюсы.
   Синяя линия там ровно 0 — за стену влияние не проходит. Это аксиома A1 в панели под графиком.

2. **Разбавление.** Оранжевая линия (контрольная цепочка, текущая модель) лежит заметно ниже
   базовой, хотя эти люди к Бобу отношения не имеют: минус Алисы физически оттягивает на себя
   долю массы её блужданий, равную `|w⁻|/Σ|w|`. Покрутите `|w⁻|` — просадка растёт.
   Каждая модель отложена в долях от **своей собственной** картины без минуса, так что
   уровни сравнимы: это «сколько осталось от того, что было бы без негативного ребра».

3. **Стена без дискредитации.** Снимите `обратная дискредитация`. Цепочка к Бобу всё равно
   проседает, и тем сильнее, чем ближе к Бобу: каждый теряет ровно ту долю влияния, которую
   вложил в мошенника ($q_C$). Уже это отталкивает подсеть Боба — и без единой поверхности атаки.

4. **Обратная дискредитация.** Включите обратно и двигайте `λ`. Узлы уходят в минус там, где
   $q_C > 1/(1+\lambda)$. При `λ=1` это $q > 0.5$: «больше половины твоего доверия ведёт к Бобу».

5. **Радиус вины.** Включите `вина только прямому поручителю`: вместо плавного провала по всей
   цепочке остаётся одна яма у `aK`. Ближние друзья Алисы не задеты — ценой того, что цепочку
   сообщников штраф не достаёт. Это ось «кого наказывать», к наследованию отношения не имеет.

6. **Ловушка нормировки.** Включите её при `подопечных у Боба` ≥ 1 — зелёная контрольная цепочка
   поднимется **выше** базовой линии: Алиса забанила Боба, а непричастные получили бесплатный
   бонус, потому что знаменатель `total_hits` усох вместе с поглощёнными блужданиями. A6 краснеет.
   При `подопечных = 0` ловушка не проявляется: умирать нечему.

7. **Мягкая стена — и как две безопасные поблажки ломают аксиому вместе.** Опустите `s` ниже 1:
   Боб сохраняет $(1-d)$ долю прохода и подкармливает через неё подопечных. Само по себе это
   A1 не нарушает — сдвиг остаётся 0.00%, потому что при фиксированном знаменателе лишний поток
   к подопечным не задевает третьих лиц. Но включите вдобавок `ловушку нормировки` — и A1
   краснеет (≈1.5%): теперь подопечные попадают в знаменатель, и Боб снова двигает чужие оценки
   тем, кого решил похвалить. Каждая поблажка по отдельности безобидна, вместе — дыра.

8. **Наследование недоверия (Guha 1-step; фаза 2, отложена — #86).** Включите `наследованное недоверие` — теперь минус
   ставит не Алиса, а её знакомый `cj`, и Алиса подхватывает его на один шаг. Двигайте
   `позиция обвинителя`: чем он дальше от Алисы, тем слабее стена ($d = s\,\alpha^{\,j-1}$)
   и тем выше поднимается синяя кривая обратно к базовой. Обвинитель на графике помечен
   стрелкой и, что важно, **сам не наказан** — он на контрольной цепочке, ни в один
   отравленный префикс не попадает. Так работает отдельный проход (D6). В варианте «состояние
   внутри блуждания» стену пришлось бы подхватывать в `cj`, и `cj` стоял бы в префиксе каждого
   отравленного им блуждания; здесь от `cj` до Боба пути нет, и такой вариант не отравил бы ничего.

9. **Донос в текущей модели стоит доносчику.** При включённом наследовании посмотрите на
   оранжевую линию: сразу за `cj` она обрывается вниз. В текущей модели минус — обычное ребро,
   оно оттягивает долю `|w⁻|/(1+|w⁻|)` массы блужданий `cj`, и его собственные друзья ниже по цепочке
   теряют влияние просто потому, что он кого-то обвинил. В новой модели этого нет:
   зелёная линия проходит через `cj` без излома. Заодно видно, что там наследование вообще
   не одношаговое — блуждание сворачивает в минус из любой точки, куда сумело дойти.

## Разбор: что именно делает обратная дискредитация

Штраф едет **не вперёд от Боба, а назад по тому самому блужданию, которое в Боба упёрлось**:

```
блуждание завершилось само:  Алиса → a1 → a2 → a3           +1 каждому: a1, a2, a3
блуждание поглощено стеной:  Алиса → a1 → … → a6 → [Боб]    −λ·b_i каждому: a1…a6, Боб
```

Узел получает не «плюс за визит», а плюс или минус — **смотря чем кончилось блуждание, которое через
него прошло**. Прошло через тебя и мирно завершилось — ты хороший проводник доверия. Прошло через
тебя и упёрлось в мошенника — значит, ты к нему и привёл.

Отсюда формула. За одно блуждание узел C посещается $n_C$ раз; доля $(1-q_C)$ этих визитов приходится
на «хорошие» блуждания, доля $q_C$ — на отравленные:

$$\text{score}(C)\;=\;\underbrace{n_C(1-q_C)\cdot(+1)}_{\text{зачтено}}\;+\;\underbrace{n_C\,q_C\cdot(-\lambda)}_{\text{вина}}\;=\;n_C\bigl(1-(1+\lambda)q_C\bigr)$$

**Здесь два разных механизма, и их легко перепутать:**

1. **Стена** (работает даже при `λ=0`) — это *неначисление*. Отравленное блуждание просто не приносит
   плюсов, и C теряет долю $q_C$ своих хитов. Наказание пассивное: «ты потратил часть своего
   влияния впустую».
2. **Обратная дискредитация** (`λ>0`) — это *активное вычитание* поверх. Те же блуждания не просто
   не дают плюс, а дают минус. Поэтому в скобках стоит $(1+\lambda)$, а не $(1-\lambda)$:
   единица — неначисленный плюс, $\lambda$ — доначисленный минус.

При `λ=1` узел уходит в ноль ровно когда $q_C = 0.5$ — половина его исходящего доверия ведёт
к Бобу. При `λ=0` в ноль он не уйдёт никогда, только просядет.

`γ` управляет тем, поровну ли делится вина: вклад узла — $-\lambda\gamma^{\,\text{расстояние до Боба}}$. Сам Боб стоит на расстоянии 0
и получает полный вес $b = 1$; эго освобождено.
При `γ=1` весь префикс виноват одинаково, при `γ→0` вина стягивается к прямому поручителю —
предельный случай и есть галочка «вина только прямому поручителю» (радиус вины = 1).
Это **не** имеет отношения к одношаговому наследованию недоверия по Guha: там «один шаг»
считается вперёд по доверию и отвечает на вопрос, чей минус вообще попадёт в рамку Алисы.

Ячейка ниже разбирает итоговую высоту каждого столбца на эти две части и показывает несколько
конкретных блужданий. Управляется теми же слайдерами, что и главный график (прокрутите вверх).
Нормировка здесь всегда фиксированная — разбор про $\lambda$, а не про ловушку.

In [5]:
import random

def breakdown(alpha=ALPHA, K=K0, m=M0, s=1.0, lam=0.6, blame_one_hop=False, gamma=0.8,
              backward=True, inherit=False, acc=2):
    lam = lam if backward else 0.0
    d = effective_d(alpha, K, m, s, inherit, acc)
    B, La, Lc = dims(K, m)
    xs = list(range(1, La + 1))
    bw = blame_weights(K, m, lam, blame_one_hop, gamma)
    ref = 0.5 * alpha                              # доверие к первому другу без минуса

    base, withheld, blamed = {}, {}, {}
    for i in xs:
        n = 0.5 * alpha ** i                       # визитов за блуждание, до всякого недоверия
        q = d * alpha ** (B - i) if i <= B else d  # P(блуждание будет поглощено стеной)
        base[i] = n / ref
        withheld[i] = n * q / ref                  # плюсы, которые не начислены (стена)
        blamed[i] = lam * bw[i] * n * q / ref      # минусы, доначисленные сверху (λ)
    final = {i: base[i] - withheld[i] - blamed[i] for i in xs}

    fig, ax = plt.subplots(figsize=(11.5, 5.6))
    ax.axhline(0, color='#333', lw=0.9)
    ax.axvline(B, color='#d9534f', lw=1.0, ls=':', alpha=0.8)
    ax.annotate('Боб ✗', xy=(B, max(base.values()) * 1.06), ha='center',
                fontsize=10.5, color='#d9534f', fontweight='bold')
    for i in xs:
        ax.vlines(i, final[i] + blamed[i], base[i], color='#f0ad4e', lw=11, alpha=0.85)
        ax.vlines(i, final[i], final[i] + blamed[i], color='#d9534f', lw=11, alpha=0.85)
    ax.plot(xs, [base[i] for i in xs], ls='--', lw=1.4, color='#888')
    ax.plot(xs, [final[i] for i in xs], '-o', lw=2.4, ms=6, color='#1f77b4', zorder=5)

    from matplotlib.lines import Line2D
    ax.legend(handles=[
        Line2D([], [], ls='--', color='#888', label='без минуса (базовая линия)'),
        Line2D([], [], color='#f0ad4e', lw=9, label='не зачтено — стена (даже при λ=0)'),
        Line2D([], [], color='#d9534f', lw=9, label='активная вина — обратная дискредитация (λ)'),
        Line2D([], [], color='#1f77b4', marker='o', lw=2.4, label='итоговая оценка'),
    ], loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=2, fontsize=9, frameon=False)
    ax.set_xticks(xs)
    ax.set_xlabel('позиция в цепочке')
    ax.set_ylabel('доверие эго')
    ax.set_title('Из чего складывается потеря: стена отдельно, вина отдельно', fontsize=12.5)
    ax.set_ylim(min(-0.05, min(final.values()) * 1.25), max(base.values()) * 1.16)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    lbl = lambda i: 'Боб' if i == B else (f'a{i}' if i < B else f't{i - B}')
    if lam <= 0.0:
        print('обратная дискредитация выключена (λ=0): отравленное блуждание просто не начисляет '
              'плюсы —\nвидна только жёлтая часть, «не зачтено».\n')
    else:
        print('веса вины b_i:  '
              + '  '.join(f'{lbl(i)} {bw[i]:.2f}' for i in range(1, B + 1))
              + ('   (радиус вины = 1: винится только прямой поручитель)' if blame_one_hop else
                 f'   (γ={gamma:.2f} — затухание по расстоянию до Боба)'))
        print(f'вклад отравленного блуждания в узел i:  −λ·b_i = −{lam:.2f}·b_i\n')
    print(sample_walks(alpha, K, m, d, lam, blame_one_hop, gamma))


def sample_walks(alpha, K, m, d, lam, blame_one_hop, gamma, seed=7):
    """Несколько конкретных блужданий и то, что они начисляют."""
    rnd = random.Random(seed)
    B, La, Lc = dims(K, m)
    bw = blame_weights(K, m, lam, blame_one_hop, gamma)
    buckets = {'poisoned': [], 'alive': [], 'ctl': []}
    target = {'poisoned': 2, 'alive': 2, 'ctl': 1}
    for _ in range(4000):
        if rnd.random() > alpha:
            continue
        a = rnd.random() < 0.5
        node, path, poisoned = 0, [], False
        while True:
            node += 1
            path.append(node)
            if a and node == B and rnd.random() < d:
                poisoned = True
                break
            if node >= (La if a else Lc) or rnd.random() > alpha:
                break
        key = 'poisoned' if poisoned else ('alive' if a else 'ctl')
        if len(buckets[key]) < target[key] and path not in buckets[key]:
            buckets[key].append(path)          # без дублей: одинаковые блуждания не информативны
        if all(len(buckets[k]) >= target[k] for k in target):
            break

    lbl = lambda i, a: ('Боб' if i == B else (f'a{i}' if i < B else f't{i - B}')) if a else f'c{i}'
    out = ['разбор нескольких блужданий:']
    for key in ('alive', 'poisoned', 'ctl'):
        for path in buckets[key]:
            a = key != 'ctl'
            chain = ' → '.join(['Алиса'] + [lbl(i, a) for i in path])
            if key == 'poisoned' and lam <= 0.0:
                out.append(f'  ⛔ {chain}\n       поглощено Бобом  →  ничего не начислено: '
                           'плюсов нет, минусов тоже (λ=0, работает только стена)')
            elif key == 'poisoned':
                gives = '  '.join(f'{lbl(i, a)} {-lam * bw[i]:+.2f}' for i in path)
                out.append(f'  ⛔ {chain}\n       поглощено Бобом  →  {gives}')
            else:
                out.append(f'  ✓  {chain}\n       завершилось само →  +1 каждому')
    return '\n'.join(out)


display(w.interactive_output(breakdown, BIND))   # те же виджеты, что и у главного графика

Output()

In [6]:
import random

def monte_carlo(alpha, K, m, d, lam, blame_one_hop, gamma, trap, W=60000, seed=0):
    """Честная симуляция блужданий — проверка, что аналитика выше не врёт."""
    rnd = random.Random(seed)
    B, La, Lc = dims(K, m)
    N = max(La, Lc)
    pos = [0.0] * (N + 1); neg = [0.0] * (N + 1); ctl = [0.0] * (N + 1)
    ego = 0.0
    for _ in range(W):
        ego += 1
        if rnd.random() > alpha:
            continue
        chain_a = rnd.random() < 0.5
        end = La if chain_a else Lc
        node, path, poisoned = 0, [], False
        while True:
            node += 1
            path.append(node)
            if chain_a and node == B and rnd.random() < d:
                poisoned = True
                break
            if node >= end or rnd.random() > alpha:
                break
        for i in path:
            if not chain_a:
                ctl[i] += 1
            elif poisoned:
                neg[i] += 1
            else:
                pos[i] += 1
    b = blame_weights(K, m, lam, blame_one_hop, gamma)
    A = [(pos[i] - lam * b[i] * neg[i]) / W for i in range(N + 1)]
    C = [ctl[i] / W for i in range(N + 1)]
    if trap:
        Z = (ego + sum(pos) + sum(neg) + sum(ctl)) / W
        A = [x / Z for x in A]; C = [x / Z for x in C]
    return A, C


def monte_carlo_current(alpha, K, m, wneg, inherit, acc, W=60000, seed=1):
    """Симуляция текущей семантики: негативное ребро проходимо, суффикс после него штрафуется.

    Контрольная цепочка живёт на id 100+i, цепочка к Бобу — на 1..La (Боб = B).
    """
    rnd = random.Random(seed)
    B, La, Lc = dims(K, m)
    j = accuser_pos(K, m, acc)
    adj = {0: [(1, 1.0), (101, 1.0)] + ([] if inherit else [(B, -wneg)])}
    for i in range(1, La + 1):
        adj[i] = [(i + 1, 1.0)] if i < La else []
    for i in range(1, Lc + 1):
        adj[100 + i] = [(100 + i + 1, 1.0)] if i < Lc else []
    if inherit:
        adj[100 + j].append((B, -wneg))

    N = max(La, Lc)
    pos = {}; neg = {}
    pos[0] = W                                    # эго попадает в каждое блуждание
    for _ in range(W):
        node, path, neg_at = 0, [], None
        while rnd.random() <= alpha:
            edges = adj[node]
            if neg_at is not None:
                edges = [e for e in edges if e[1] > 0]      # в негативном режиме — только плюсы
            if not edges:
                break
            tot = sum(abs(x) for _, x in edges)
            r = rnd.random() * tot
            for tgt, x in edges:
                r -= abs(x)
                if r <= 0:
                    break
            path.append(tgt)
            if x < 0 and neg_at is None:
                neg_at = len(path) - 1
            node = tgt
        cut = len(path) if neg_at is None else neg_at
        for i in path[:cut]:
            pos[i] = pos.get(i, 0) + 1
        for i in path[cut:]:
            neg[i] = neg.get(i, 0) + 1
    Z = sum(pos.values()) + sum(neg.values())     # хит эго идёт в знаменатель...
    A = [(pos.get(i, 0) - neg.get(i, 0)) / Z if i else 0.0    # ...но не в отчёт: аналитика
         for i in range(N + 1)]                               # тоже не считает score самой Алисы
    C = [(pos.get(100 + i, 0) - neg.get(100 + i, 0)) / Z if i else 0.0
         for i in range(N + 1)]
    return A, C


worst = 0.0
for blame_one_hop_ in (False, True):
    for lam_ in (0.0, 0.6):
        for trap_ in (False, True):
            for d_ in (1.0, 0.5):
                an = new_model(0.85, 6, 3, d_, lam_, blame_one_hop_, 0.8, trap_)
                mc = monte_carlo(0.85, 6, 3, d_, lam_, blame_one_hop_, 0.8, trap_)
                worst = max(worst, max(abs(x - y)
                                       for p in zip(an, mc) for x, y in zip(*p)))
print(f'новая модель — макс. расхождение аналитики и Монте-Карло (W=60k): {worst:.4f}')

worst_c = 0.0
for inherit_ in (False, True):
    for acc_ in (1, 4):
        for wneg_ in (0.5, 1.5):
            an = current_model(0.85, 6, 3, wneg_, inherit_, acc_)
            mc = monte_carlo_current(0.85, 6, 3, wneg_, inherit_, acc_)
            worst_c = max(worst_c, max(abs(x - y)
                                       for p in zip(an, mc) for x, y in zip(*p)))
print(f'текущая модель — макс. расхождение (W=60k):                      {worst_c:.4f}')
print('\nоба — в пределах шума выборки')

новая модель — макс. расхождение аналитики и Монте-Карло (W=60k): 0.0039
текущая модель — макс. расхождение (W=60k):                      0.0010

оба — в пределах шума выборки


## Перенос в код

| В ноутбуке | В репозитории |
|---|---|
| поглощение на Бобе, $q_i$ | `graph.rs::generate_walk_segment` — `random_neighbor` всегда `positive_only`, поглощение по ego-относительному множеству $D_A$ |
| `poisoned` вместо негативного суффикса | `random_walk.rs` — `negative_segment_start: Option<usize>` → `absorbed_at: Option<NodeId>`; `positive_subsegment` / `negative_subsegment` схлопываются |
| `blame_weights` | новый шаг учёта в `rank.rs::calculate` |
| фиксированный знаменатель | `rank.rs::get_node_score`, строки 108–112 — `total_hits` → `walks_per_ego` |
| инвалидация при добавлении минуса | `walk_storage::get_visits_through_node(B)` уже отдаёт нужные блуждания: обрезать по Бобу, префикс перекинуть из `pos_hits` в `neg_hits` |
| минус = отрицательный вес ребра | `mr_put_edge` без изменения сигнатуры; $d = \min(|w|, 1)$ |
| стены приватны | `mr_graph` / `mr_neighbors` не отдают отрицательные рёбра ни при каком `positive_only` |

Аксиомы A1 и A6 из панели под графиком — прямые кандидаты в тесты; к ним стоит добавить
A2 (узел, достижимый только через Боба, получает 0), A5 (снятие ребра `C→B` восстанавливает C)
A7 (суммарная потеря в рамке Алисы ≤ $(1+\lambda)\times$ поток в Боба) и A9 (в фазе 1 стена
Алисы не меняет ни одной чужой рамки). Полный список тестов — `NEGATIVE_EDGES_FEATURE.md`, §7.

**Чего этот ноутбук не показывает** — цепочка для этого слишком проста: наследование недоверия
при реалистичном ветвлении (здесь обвинитель один), «проблему хаба» (накопление $q$ у друга-хаба при
десятках минусов; острее, когда мягкие стены повсеместны) и атаку «отравленной приманки». Первые две
проверяются только на реальном дампе.